# Análise de Perfis de Consumo Elétrico — Aveiro
Análise exploratória completa sobre o dataset pré-processado, cobrindo:
- Perfis horários médios (dia útil vs. fim de semana)
- Decomposição sazonal (tendência, sazonalidade, resíduos)
- Comparação Verão vs. Inverno
- Análise de picos de consumo
- Consumo Diurno vs. Noturno

In [1]:
# =============================================================
# CÉLULA 1 — IMPORTAÇÕES
# =============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


## 1. Carregar o Dataset Pré-processado

In [7]:
df = pd.read_csv("serie_projeccao_consumo_horario_2024_2025_aveiro.csv", parse_dates=["data_hora"])
df = df.sort_values("data_hora").reset_index(drop=True)

# Garantir colunas auxiliares caso não existam
if "hora" not in df.columns:
    df["hora"] = df["data_hora"].dt.hour
if "mes" not in df.columns:
    df["mes"]  = df["data_hora"].dt.month
if "dia_semana" not in df.columns:
    df["dia_semana"] = df["data_hora"].dt.dayofweek
if "fim_de_semana" not in df.columns:
    df["fim_de_semana"] = (df["dia_semana"] >= 5).astype(int)
if "estacao" not in df.columns:
    def estacao(mes):
        if mes in [12,1,2]: return "Inverno"
        elif mes in [3,4,5]: return "Primavera"
        elif mes in [6,7,8]: return "Verão"
        else: return "Outono"
    df["estacao"] = df["mes"].apply(estacao)

print(f"Dataset carregado: {len(df):,} registos | {df['data_hora'].min().date()} → {df['data_hora'].max().date()}")
print(f"Colunas: {list(df.columns)}")


Dataset carregado: 17,544 registos | 2024-01-01 → 2025-12-31
Colunas: ['data_hora', 'ano', 'mes', 'dia', 'hora', 'energia_ativa_kwh', 'chave_mes_ano', 'dia_da_semana', 'dia_semana', 'fim_de_semana', 'estacao']


## 2. Perfis Horários Médios
O perfil horário médio revela os padrões típicos de consumo ao longo do dia.
A distinção entre **dias úteis** e **fins de semana** é fundamental: nos dias úteis
observam-se dois picos marcados (manhã e fim do dia), enquanto ao fim de semana
o consumo é mais uniforme e deslocado para horas mais tardias.

In [9]:
# Perfil horário médio — dia útil vs. fim de semana
perfil_util    = df[df["fim_de_semana"] == 0].groupby("hora")["energia_ativa_kwh"].mean()
perfil_fds     = df[df["fim_de_semana"] == 1].groupby("hora")["energia_ativa_kwh"].mean()
perfil_global  = df.groupby("hora")["energia_ativa_kwh"].mean()

fig_perfil = go.Figure()

fig_perfil.add_trace(go.Scatter(
    x=perfil_util.index, y=perfil_util.values,
    mode="lines+markers", name="Dia Útil",
    line=dict(color="#0284C7", width=2.5),
    marker=dict(size=5)
))
fig_perfil.add_trace(go.Scatter(
    x=perfil_fds.index, y=perfil_fds.values,
    mode="lines+markers", name="Fim de Semana",
    line=dict(color="#EA580C", width=2.5, dash="dash"),
    marker=dict(size=5)
))
fig_perfil.add_trace(go.Scatter(
    x=perfil_global.index, y=perfil_global.values,
    mode="lines", name="Média Global",
    line=dict(color="#94A3B8", width=1.5, dash="dot")
))

# Sombrear período noturno (22h–06h)
fig_perfil.add_vrect(x0=0, x1=6,   fillcolor="#E2E8F0", opacity=0.4, layer="below", line_width=0)
fig_perfil.add_vrect(x0=22, x1=23, fillcolor="#E2E8F0", opacity=0.4, layer="below", line_width=0)

fig_perfil.update_layout(
    title="Perfil Horário Médio de Consumo — Aveiro",
    xaxis_title="Hora do Dia",
    yaxis_title="Energia Ativa Média (kWh)",
    xaxis=dict(tickmode="linear", tick0=0, dtick=1, showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=420
)
fig_perfil.show()

# Resumo estatístico
hora_pico_util = perfil_util.idxmax()
hora_pico_fds  = perfil_fds.idxmax()
print(f"Hora de pico — Dia Útil: {hora_pico_util}h | Fim de Semana: {hora_pico_fds}h")


Hora de pico — Dia Útil: 19h | Fim de Semana: 19h


## 3. Decomposição Sazonal da Série Temporal
A decomposição aditiva clássica separa a série em três componentes:
- **Tendência:** evolução de longo prazo do consumo médio
- **Sazonalidade:** padrão periódico repetido (ciclo semanal de 7 dias)
- **Resíduos:** variações irregulares não explicadas pelos componentes anteriores

Utiliza-se a agregação diária para reduzir ruído horário e tornar a decomposição mais legível.

In [11]:
# Agregar para série diária
df_diario = df.set_index("data_hora")["energia_ativa_kwh"].resample("D").sum().reset_index()
df_diario.columns = ["data", "energia_diaria"]
df_diario = df_diario.set_index("data")

# Decomposição aditiva — period=7 (ciclo semanal)
decomp = seasonal_decompose(df_diario["energia_diaria"], model="additive", period=7)

fig_decomp = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=(
        "Sinal Observado — Consumo Diário Total (kWh)",
        "Tendência — Evolução de Longo Prazo",
        "Sazonalidade — Padrão Semanal (período = 7 dias)",
        "Resíduos — Variações Irregulares e Anomalias"
    )
)

fig_decomp.add_trace(go.Scatter(x=decomp.observed.index,  y=decomp.observed,
    mode="lines", line=dict(color="#475569", width=1.2), name="Observado"), row=1, col=1)
fig_decomp.add_trace(go.Scatter(x=decomp.trend.index,     y=decomp.trend,
    mode="lines", line=dict(color="#2563EB", width=2),   name="Tendência"), row=2, col=1)
fig_decomp.add_trace(go.Scatter(x=decomp.seasonal.index,  y=decomp.seasonal,
    mode="lines", line=dict(color="#0D9488", width=1.5), name="Sazonalidade"), row=3, col=1)
fig_decomp.add_trace(go.Scatter(x=decomp.resid.index,     y=decomp.resid,
    mode="markers", marker=dict(color="#EA580C", size=3), name="Resíduos"), row=4, col=1)

# Linha zero nos resíduos
fig_decomp.add_hline(y=0, line_dash="dot", line_color="#CBD5E1", row=4, col=1)

fig_decomp.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    height=700, showlegend=False, margin=dict(t=40, b=20)
)
for i in range(1, 5):
    fig_decomp.update_xaxes(showgrid=False, linecolor="#E2E8F0", row=i, col=1)
    fig_decomp.update_yaxes(showgrid=True,  gridcolor="#F1F5F9", row=i, col=1)
fig_decomp.show()

# Peso relativo de cada componente
var_tend  = decomp.trend.dropna().var()
var_saz   = decomp.seasonal.var()
var_resid = decomp.resid.dropna().var()
var_total = var_tend + var_saz + var_resid
print(f"Variância explicada — Tendência: {var_tend/var_total*100:.1f}% | "
      f"Sazonalidade: {var_saz/var_total*100:.1f}% | Resíduos: {var_resid/var_total*100:.1f}%")


Variância explicada — Tendência: 75.2% | Sazonalidade: 0.0% | Resíduos: 24.8%


## 4. Comparação Verão vs. Inverno
A comparação sazonal entre Verão (Jun–Ago) e Inverno (Dez–Fev) evidencia
as diferenças de magnitude e de distribuição horária do consumo, reflexo
das diferentes necessidades de climatização e dos padrões de luminosidade.

In [13]:
df_verao   = df[df["estacao"] == "Verão"]
df_inverno = df[df["estacao"] == "Inverno"]

perfil_verao   = df_verao.groupby("hora")["energia_ativa_kwh"].mean()
perfil_inverno = df_inverno.groupby("hora")["energia_ativa_kwh"].mean()

# --- Gráfico 1: Perfil horário Verão vs. Inverno ---
fig_vi = go.Figure()

fig_vi.add_trace(go.Scatter(
    x=perfil_inverno.index, y=perfil_inverno.values,
    mode="lines+markers", name="Inverno",
    line=dict(color="#2563EB", width=2.5), marker=dict(size=5),
    fill="tozeroy", fillcolor="rgba(37,99,235,0.08)"
))
fig_vi.add_trace(go.Scatter(
    x=perfil_verao.index, y=perfil_verao.values,
    mode="lines+markers", name="Verão",
    line=dict(color="#EA580C", width=2.5), marker=dict(size=5),
    fill="tozeroy", fillcolor="rgba(234,88,12,0.08)"
))

fig_vi.add_vrect(x0=0,  x1=6,  fillcolor="#E2E8F0", opacity=0.35, layer="below", line_width=0)
fig_vi.add_vrect(x0=22, x1=23, fillcolor="#E2E8F0", opacity=0.35, layer="below", line_width=0)

fig_vi.update_layout(
    title="Perfil Horário Médio — Verão vs. Inverno",
    xaxis_title="Hora do Dia", yaxis_title="Energia Ativa Média (kWh)",
    xaxis=dict(tickmode="linear", tick0=0, dtick=1, showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=420
)
fig_vi.show()

# --- Gráfico 2: Box plot distribuição diária por estação ---
df_box = df[df["estacao"].isin(["Verão", "Inverno", "Primavera", "Outono"])].copy()
df_box_diario = df_box.groupby([df_box["data_hora"].dt.date, "estacao"])["energia_ativa_kwh"].sum().reset_index()
df_box_diario.columns = ["data", "estacao", "energia_diaria"]

ordem_estacoes = ["Inverno", "Primavera", "Verão", "Outono"]
cores_estacoes = {"Inverno": "#2563EB", "Primavera": "#16A34A", "Verão": "#EA580C", "Outono": "#D97706"}

fig_box = go.Figure()
for est in ordem_estacoes:
    dados_est = df_box_diario[df_box_diario["estacao"] == est]["energia_diaria"]
    fig_box.add_trace(go.Box(
        y=dados_est, name=est,
        marker_color=cores_estacoes[est],
        boxmean="sd"
    ))

fig_box.update_layout(
    title="Distribuição do Consumo Diário por Estação do Ano",
    yaxis_title="Consumo Diário Total (kWh)",
    plot_bgcolor="white", paper_bgcolor="white",
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    height=420, showlegend=False
)
fig_box.show()

# Resumo numérico
print("Consumo médio por estação (kWh/hora):")
print(df.groupby("estacao")["energia_ativa_kwh"].mean().sort_values(ascending=False).round(2))


Consumo médio por estação (kWh/hora):
estacao
Inverno      44.03
Outono       35.83
Primavera    35.30
Verão        25.99
Name: energia_ativa_kwh, dtype: float64


## 5. Análise de Picos de Consumo
Identificam-se os momentos de maior exigência sobre a rede, definindo pico
como qualquer registo que supere o **percentil 95** do consumo horário.
Esta análise é determinante para avaliar a capacidade de uma comunidade
de energia em reduzir a pressão sobre a rede nos períodos críticos.

In [15]:
# Definir limiar de pico (percentil 95)
limiar_pico = df["energia_ativa_kwh"].quantile(0.95)
df["e_pico"] = df["energia_ativa_kwh"] > limiar_pico

print(f"Limiar de pico (P95): {limiar_pico:.2f} kWh")
print(f"Horas em pico: {df['e_pico'].sum():,} ({df['e_pico'].mean()*100:.1f}% do total)")

# --- Gráfico 1: Distribuição dos picos por hora do dia ---
picos_por_hora = df[df["e_pico"]].groupby("hora").size()

fig_picos_hora = go.Figure(go.Bar(
    x=picos_por_hora.index, y=picos_por_hora.values,
    marker_color=["#DC2626" if v == picos_por_hora.max() else "#0284C7" for v in picos_por_hora.values],
    name="Picos"
))
fig_picos_hora.update_layout(
    title="Frequência de Picos de Consumo por Hora do Dia (P95)",
    xaxis_title="Hora do Dia", yaxis_title="Nº de Ocorrências",
    xaxis=dict(tickmode="linear", tick0=0, dtick=1, showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    plot_bgcolor="white", paper_bgcolor="white", height=380
)
fig_picos_hora.show()

# --- Gráfico 2: Picos por estação do ano ---
picos_estacao = df[df["e_pico"]].groupby("estacao").size().reindex(ordem_estacoes)

fig_picos_est = go.Figure(go.Bar(
    x=picos_estacao.index, y=picos_estacao.values,
    marker_color=[cores_estacoes[e] for e in picos_estacao.index]
))
fig_picos_est.update_layout(
    title="Frequência de Picos de Consumo por Estação do Ano",
    xaxis_title="Estação", yaxis_title="Nº de Ocorrências",
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    plot_bgcolor="white", paper_bgcolor="white", height=380
)
fig_picos_est.show()

# --- Gráfico 3: Top 10 horas de maior consumo absoluto ---
top10 = df.nlargest(10, "energia_ativa_kwh")[["data_hora", "energia_ativa_kwh", "estacao", "hora"]]
print("Top 10 registos de consumo máximo:")
print(top10.to_string(index=False))


Limiar de pico (P95): 84.65 kWh
Horas em pico: 875 (5.0% do total)


Top 10 registos de consumo máximo:
          data_hora  energia_ativa_kwh estacao  hora
2025-12-05 09:00:00         158.266079 Inverno     9
2025-12-08 13:00:00         158.244492 Inverno    13
2024-12-05 09:00:00         155.162823 Inverno     9
2024-12-08 13:00:00         155.141659 Inverno    13
2025-12-11 03:00:00         154.657867 Inverno     3
2025-12-27 10:00:00         154.654808 Inverno    10
2025-01-05 09:00:00         152.178922 Inverno     9
2025-01-08 13:00:00         152.158166 Inverno    13
2024-12-11 03:00:00         151.625360 Inverno     3
2024-12-27 10:00:00         151.622361 Inverno    10


## 6. Consumo Diurno vs. Noturno
A divisão entre período diurno (06h–22h) e noturno (22h–06h) permite quantificar
a assimetria de consumo ao longo do dia, um parâmetro crítico para dimensionar
sistemas de armazenamento de energia (baterias) em comunidades energéticas.

In [16]:
# Classificar período diurno/noturno
df["periodo"] = df["hora"].apply(lambda h: "Diurno (06h–22h)" if 6 <= h < 22 else "Noturno (22h–06h)")

# --- Totais globais ---
totais = df.groupby("periodo")["energia_ativa_kwh"].sum()
pct_diurno  = totais["Diurno (06h–22h)"]  / totais.sum() * 100
pct_noturno = totais["Noturno (22h–06h)"] / totais.sum() * 100
print(f"Consumo Diurno : {pct_diurno:.1f}%")
print(f"Consumo Noturno: {pct_noturno:.1f}%")

# --- Gráfico 1: Donut global ---
fig_donut = go.Figure(go.Pie(
    labels=totais.index, values=totais.values,
    hole=0.55,
    marker_colors=["#F59E0B", "#1E3A5F"],
    textinfo="label+percent",
    hovertemplate="%{label}: %{value:.0f} kWh (%{percent})<extra></extra>"
))
fig_donut.update_layout(
    title="Repartição Global do Consumo — Diurno vs. Noturno",
    plot_bgcolor="white", paper_bgcolor="white", height=380
)
fig_donut.show()

# --- Gráfico 2: Diurno vs Noturno por estação ---
dn_estacao = (
    df.groupby(["estacao", "periodo"])["energia_ativa_kwh"]
    .mean()
    .unstack()
    .reindex(ordem_estacoes)
)

fig_dn = go.Figure()
fig_dn.add_trace(go.Bar(
    x=dn_estacao.index, y=dn_estacao["Diurno (06h–22h)"],
    name="Diurno", marker_color="#F59E0B"
))
fig_dn.add_trace(go.Bar(
    x=dn_estacao.index, y=dn_estacao["Noturno (22h–06h)"],
    name="Noturno", marker_color="#1E3A5F"
))
fig_dn.update_layout(
    title="Consumo Médio Diurno vs. Noturno por Estação",
    xaxis_title="Estação", yaxis_title="Energia Média por Hora (kWh)",
    barmode="group",
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=400
)
fig_dn.show()

# --- Gráfico 3: Evolução mensal do rácio diurno/noturno ---
dn_mensal = (
    df.groupby(["mes", "periodo"])["energia_ativa_kwh"]
    .mean()
    .unstack()
)
dn_mensal["racio"] = dn_mensal["Diurno (06h–22h)"] / dn_mensal["Noturno (22h–06h)"]

meses_nomes = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

fig_racio = go.Figure(go.Scatter(
    x=[meses_nomes[m-1] for m in dn_mensal.index],
    y=dn_mensal["racio"].values,
    mode="lines+markers",
    line=dict(color="#7C3AED", width=2.5),
    marker=dict(size=7, color="#7C3AED"),
    fill="tozeroy", fillcolor="rgba(124,58,237,0.08)"
))
fig_racio.add_hline(y=1, line_dash="dot", line_color="#94A3B8",
                    annotation_text="Consumo Diurno = Noturno",
                    annotation_position="bottom right")
fig_racio.update_layout(
    title="Rácio Diurno/Noturno por Mês (valores > 1 = maior consumo diurno)",
    xaxis_title="Mês", yaxis_title="Rácio Diurno / Noturno",
    yaxis=dict(showgrid=True, gridcolor="#F1F5F9"),
    plot_bgcolor="white", paper_bgcolor="white", height=380
)
fig_racio.show()


Consumo Diurno : 73.0%
Consumo Noturno: 27.0%
